# OIBSIP Data Analytics – Level 2 – Task 5
# Predicting House Prices with Linear Regression

## 1. EDA
Check nulls, descriptive statistics and price distribution.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
df=pd.read_csv("house_prices.csv")
print(df.isnull().sum())
display(df.describe(include="all"))
sns.histplot(df["Price"],kde=True); plt.title("Distribution of House Prices"); plt.show()


## 2. Feature selection discussion
Area, location, rooms and age are selected because they are plausible predictors of house price. Location is categorical and therefore requires One-Hot Encoding.

In [ ]:
X=df[["Area_sqft","Location","Rooms","Age_years"]]; y=df["Price"]
numeric=["Area_sqft","Rooms","Age_years"]; categorical=["Location"]
prep=ColumnTransformer([("num",SimpleImputer(strategy="median"),numeric),("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),categorical)])


## 3. Correlation heatmap

In [ ]:
plt.figure(figsize=(7,5)); sns.heatmap(df[["Area_sqft","Rooms","Age_years","Price"]].corr(),annot=True,cmap="coolwarm",fmt=".2f"); plt.title("Correlation Heatmap"); plt.show()

## 4. 80/20 split and Linear Regression

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42)
model=Pipeline([("preprocessor",prep),("regressor",LinearRegression())]); model.fit(Xtr,ytr); pred=model.predict(Xte)

## 5. MSE, RMSE and R²

In [ ]:
mse=mean_squared_error(yte,pred); rmse=np.sqrt(mse); r2=r2_score(yte,pred)
print(f"MSE: {mse:,.2f}"); print(f"RMSE: {rmse:,.2f}"); print(f"R²: {r2:.4f}")

## 6. Actual vs Predicted

In [ ]:
mn,mx=min(yte.min(),pred.min()),max(yte.max(),pred.max())
plt.scatter(yte,pred,alpha=.7); plt.plot([mn,mx],[mn,mx],linestyle="--"); plt.xlabel("Actual Price"); plt.ylabel("Predicted Price"); plt.title("Actual vs Predicted House Prices"); plt.show()

## 7. Residual Plot

In [ ]:
res=yte-pred
plt.scatter(pred,res,alpha=.7); plt.axhline(0,linestyle="--"); plt.xlabel("Predicted Price"); plt.ylabel("Residual"); plt.title("Residual Plot"); plt.show()

## 8. Coefficient Analysis

In [ ]:
names=model.named_steps["preprocessor"].get_feature_names_out()
coef=pd.DataFrame({"Feature":names,"Coefficient":model.named_steps["regressor"].coef_})
display(coef.sort_values("Coefficient",ascending=False).head(5))
display(coef.sort_values("Coefficient").head(5))

## 9. Bonus – Ridge comparison

In [ ]:
ridge=Pipeline([("preprocessor",prep),("regressor",Ridge(alpha=1.0))]); ridge.fit(Xtr,ytr)
print("Linear Regression R²:",round(r2_score(yte,pred),4))
print("Ridge R²:",round(r2_score(yte,ridge.predict(Xte)),4))

## Conclusion
Report the MSE, RMSE and R², discuss the plots and identify the strongest positive/negative coefficients. Ridge is included as the optional bonus comparison.